# ViMFR: chạy V7 (6 baseline), V7b và V8 trên Colab T4
Chạy tuần tự từ trên xuống. **Runtime → Change runtime type → T4 GPU** trước khi bắt đầu.
Không chạy V7 và V8 cùng lúc: V8 đo độ trễ, V7 chiếm GPU sẽ làm sai số đo.

Chuẩn bị trên máy bạn (ở gốc dự án): `python make_eval_colab_zip.py` (code + ảnh Test) và
`python make_eval_colab_zip.py --ckpt-only --slim-ckpt` (checkpoint). Up các zip lên Drive.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

name, memory.total [MiB]
Tesla T4, 15360 MiB
2.14.1+cu130 True


## 1. Cài thư viện (xong thì Runtime → Restart session, rồi chạy tiếp từ ô 2)

In [2]:
!pip -q install "transformers>=4.57,<5" sentence-transformers timm rank_bm25 pyvi sentencepiece faiss-cpu py_vncorenlp scikit-learn pandas openpyxl tqdm
!apt-get -qq install -y default-jre-headless > /dev/null
!java -version 2>&1 | head -1
# KHÔNG pip install torch: dùng bản có sẵn của Colab. ViCLIP-OT cần torch>=2.8,<2.10, kiểm tra ở ô 1.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
[0.014s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


## 2. Nối Drive và giải nén các zip (đọc ảnh từ Drive rất chậm, nên chạy trên đĩa local)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/thesis'        # <-- SỬA: thư mục Drive chứa các zip
ZIPS = ['vimfd-eval.zip']        # code+ảnh Test, checkpoint (thêm vimfd-eval-code.zip nếu vừa sửa code)
TOP = 'vimfd-eval'
import zipfile
for z in ZIPS:
    zipfile.ZipFile(f'{DRIVE_DIR}/{z}').extractall('/content/')   # cùng thư mục gốc nên chồng lên nhau được
ROOT = f'/content/{TOP}'
%cd {ROOT}

Mounted at /content/drive
/content/vimfd-eval


In [4]:
import os
need = ['config.py','src/models/model.py','data/splits/test.jsonl','data/annotation/benchmark/queries_200.csv',
        'data/annotation/seg_utils.py','data/annotation/v7_baselines.py','data/annotation/v7b_translation_audit.py',
        'data/annotation/v8_run_vimfr.py','checkpoints/cross_encoder_best/linear_head.pt']
for p in need: print('OK ' if os.path.exists(p) else 'THIẾU', p)
!ls checkpoints/*.pt
!grep -nE "PROCESSED_DIR|CHECKPOINT_BEST|[A-Z]:\\|/home/|/Users/" config.py

OK  config.py
OK  src/models/model.py
OK  data/splits/test.jsonl
OK  data/annotation/benchmark/queries_200.csv
OK  data/annotation/seg_utils.py
OK  data/annotation/v7_baselines.py
OK  data/annotation/v7b_translation_audit.py
OK  data/annotation/v8_run_vimfr.py
OK  checkpoints/cross_encoder_best/linear_head.pt
checkpoints/best_short_24112026.pt
6:PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
7:IMAGE_DIR = os.path.join(PROCESSED_DIR, "images")
16:for _dir in [DATA_DIR, IMAGE_DIR, PROCESSED_DIR, SPLIT_DIR, CHECKPOINT_DIR, INDEX_DIR]:
71:CHECKPOINT_BEST = os.path.join(CHECKPOINT_DIR, "best_short_24112026.pt")


Nếu `config.py` có đường dẫn kiểu `C:\...` hay `/Users/...`, sửa thành đường dẫn tương đối (hoặc `/content/vimfd-eval/...`) trước khi chạy tiếp.
Nếu `THIẾU` mục nào: tạo lại zip ở máy bạn rồi giải nén lại ô 2.

In [3]:
# Bộ tách từ cho seg_utils (VnCoreNLP). Nếu seg_utils đã dùng được preprocess_query của serving thì ô tải model không cần.
import os, py_vncorenlp
os.makedirs('/content/vncorenlp', exist_ok=True)
py_vncorenlp.download_model(save_dir='/content/vncorenlp')
os.environ['VNCORENLP_DIR'] = '/content/vncorenlp'
!PYTHONPATH={ROOT}:{ROOT}/data/annotation python -c "from seg_utils import segment_text, segmenter_name; print(segmenter_name()); print(segment_text('áo khoác vải dù màu xanh rêu'))"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-24 19:34:59 INFO  WordSegmenter:24 - Loading Word Segmentation model
src.serving.text_utils.preprocess_query (VnCoreNLP, đúng pipeline serving)
áo_khoác vải dù màu xanh rêu


## 3. V7: bản dịch trước, kiểm tra mất ngữ nghĩa, rồi các hệ thống còn lại

In [5]:
!python data/annotation/v7_baselines.py --sample5          # thử 5 truy vấn, đọc top-3 bằng mắt

Truy vấn: 5 | corpus Test: 1449  [SAMPLE 5]
Device: cuda
I0000 00:00:1795549057.801899    3284 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795549060.431771    3284 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-24 19:37:51 INFO  WordSegmenter:24 - Loading Word Segmentation model

=== tfidf ===
  ✅ tfidf: 250 dòng hợp lệ (5 truy vấn)
  Đọc bằng mắt top-3 (mục 6.6):
   Q001 'quần boxer vải

In [6]:
!python data/annotation/v7_baselines.py --only clip_vi2en  # tạo translations.csv (dịch MỘT lần)
!python data/annotation/v7b_translation_audit.py

Truy vấn: 200 | corpus Test: 1449
Device: cuda
I0000 00:00:1795549203.548414    4114 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795549207.193322    4114 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-24 19:40:12 INFO  WordSegmenter:24 - Loading Word Segmentation model

=== clip_vi2en ===
  translations.csv thiếu một số query_id → dịch lại toàn bộ
  Dịch 200 truy vấn bằng vinai/vinai-tra

In [8]:
!python data/annotation/v7_baselines.py --only tfidf,bm25,bienc_text,fashionclip_vi2en

Truy vấn: 200 | corpus Test: 1449
Device: cuda
I0000 00:00:1795549518.493418    5477 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795549521.177770    5477 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-24 19:45:26 INFO  WordSegmenter:24 - Loading Word Segmentation model

=== tfidf ===
  ✅ tfidf: 10000 dòng hợp lệ (200 truy vấn)
  Đọc bằng mắt top-3 (mục 6.6):
   Q164 'quần short bò ống rộ

In [10]:
# ViCLIP-OT chạy riêng: nếu lỗi nạp, các run khác vẫn còn nguyên
!python data/annotation/v7_baselines.py --only viclip_ot

Truy vấn: 200 | corpus Test: 1449
Device: cuda
I0000 00:00:1795549977.868572    7563 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795549980.590344    7563 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-24 19:53:06 INFO  WordSegmenter:24 - Loading Word Segmentation model

=== viclip_ot ===
  Mã hóa 1449 ảnh (thiếu ảnh: 0)...
processor_config.json: 100% 289/289 [00:00<00:00, 1.80MB/s]
prepr

## 4. V8: dual-encoder + Cross-Encoder + độ trễ (T4). Đừng chạy V7 khi đang chạy ô này

In [11]:
import os
# os.environ['VIMFD_DE_CKPT'] = 'checkpoints/best_24112026.pt'   # bỏ chú thích nếu cần chỉ rõ
!python data/annotation/v8_run_vimfr.py --sample5

Device: cuda
GPU: Tesla T4
Số truy vấn: 5  [SAMPLE 5]
Test corpus: 1449 sản phẩm

Load dual-encoder: /content/vimfd-eval/checkpoints/best_short_24112026.pt  (sửa lần cuối 2026-11-24 19:33)
  LƯU Ý: checkpoint này PHẢI được train trên split hiện tại (11.571/1.450/1.449).
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-24 19:54:19 INFO  WordSegmenter:24 - Loading Word Segmentation model
  Bộ tách từ truy vấn: src.serving.text_utils.preprocess_query (VnCoreNLP, đúng pipeline serving)
I0000 00:00:1795550060.583683    7947 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA,

In [12]:
!python data/annotation/v8_run_vimfr.py --rebuild-index

Device: cuda
GPU: Tesla T4
Số truy vấn: 200
Test corpus: 1449 sản phẩm

Load dual-encoder: /content/vimfd-eval/checkpoints/best_short_24112026.pt  (sửa lần cuối 2026-11-24 19:33)
  LƯU Ý: checkpoint này PHẢI được train trên split hiện tại (11.571/1.450/1.449).
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-24 19:55:32 INFO  WordSegmenter:24 - Loading Word Segmentation model
  Bộ tách từ truy vấn: src.serving.text_utils.preprocess_query (VnCoreNLP, đúng pipeline serving)
I0000 00:00:1795550133.805097    8313 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other 

## 5. Lưu kết quả về Drive (Colab xóa đĩa local khi hết phiên)

In [13]:
OUT = f'{DRIVE_DIR}/outputs'
!mkdir -p {OUT} && rsync -a results/ {OUT}/results/ && rsync -a data/annotation/benchmark/ {OUT}/benchmark/
!ls -la results/runs

total 3796
drwxr-xr-x 2 root root   4096 Nov 24 19:56 .
drwxr-xr-x 5 root root   4096 Nov 24 19:39 ..
-rw-r--r-- 1 root root 489787 Nov 24 19:45 bienc_text.run
-rw-r--r-- 1 root root 434890 Nov 24 19:45 bm25.run
-rw-r--r-- 1 root root 492379 Nov 24 19:40 clip_vi2en.run
-rw-r--r-- 1 root root 514083 Nov 24 19:56 dual_encoder.run
-rw-r--r-- 1 root root 560256 Nov 24 19:46 fashionclip_vi2en.run
-rw-r--r-- 1 root root 442576 Nov 24 19:45 tfidf.run
-rw-r--r-- 1 root root 480523 Nov 24 19:53 viclip_ot.run
-rw-r--r-- 1 root root 444083 Nov 24 19:56 vimfr.run
